In [1]:
import pandas as pd
import numpy as np


In [2]:
from glob import glob

In [3]:
import matplotlib.pyplot as plt 
import seaborn as sns

In [4]:
import pycountry

# Carga de datos #

In [68]:
df_happiness = pd.read_csv('../DATOS/RAW/world_happiness_combined.csv', sep=";")

El dataset fue cargado especificando sep=";" porque el archivo utiliza punto y coma como delimitador en lugar de coma, algo habitual en datasets generados con configuración regional europea.

In [69]:
df_happiness.head()

,Ranking,Country,Regional indicator,Happiness score,GDP per capita,Social support,Healthy life expectancy,Freedom to make life choices,Generosity,Perceptions of corruption,Year
0,1,Switzerland,Western Europe,"7,58700","8,26132","0,96240",73,"0,99379","0,37289","0,23941",2015
1,2,Iceland,Western Europe,"7,56100","7,70416","1,00000",73,"0,93884","0,54819","0,74371",2015
2,3,Denmark,Western Europe,"7,52700","7,84114","0,97030",70,"0,96962","0,42894","0,12382",2015
3,4,Norway,Western Europe,"7,52200","8,63100","0,94917",71,"1,00000","0,43598","0,33860",2015
4,5,Canada,North America and ANZ,"7,42700","7,84595","0,94322",71,"0,94511","0,57560","0,40285",2015


In [ ]:
df_happiness.columns

In [ ]:
df_happiness.Ranking.nunique()

In [ ]:
df_happiness.Country.nunique()

In [ ]:
df_happiness.Year.unique()

In [ ]:
df_happiness.info()

In [70]:
df_happiness.columns = (
    df_happiness.columns
        .str.strip()
        .str.lower()
        .str.replace(' ', '_')
        .str.replace('.', '_', regex=False)
)
df_happiness.columns


Index(['ranking', 'country', 'regional_indicator', 'happiness_score',
       'gdp_per_capita', 'social_support', 'healthy_life_expectancy',
       'freedom_to_make_life_choices', 'generosity',
       'perceptions_of_corruption', 'year'],
      dtype='object')

In [71]:
""""Cambio a variables numericas  """

columns_to_convert = ["happiness_score", 
                      "gdp_per_capita", 
                      "social_support", 
                      "freedom_to_make_life_choices", 
                      "generosity", 
                      "perceptions_of_corruption"]


def convert_to_numeric(df, columns):
    for column in columns:
        df[column] = (
            df[column]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .str.strip()
        )
        df[column] = pd.to_numeric(df[column], errors="coerce")
    
    return df

    
df_happiness = convert_to_numeric(df_happiness, columns_to_convert)



In [ ]:
df_happiness.sample(10)

In [ ]:
df_happiness.isnull().sum()

Se decide eliminar la columna regional indicator. 

In [72]:
df_happiness.dropna(subset=["regional_indicator"], inplace=True)

In [73]:
df_happiness.describe()

,ranking,happiness_score,gdp_per_capita,social_support,healthy_life_expectancy,freedom_to_make_life_choices,generosity,perceptions_of_corruption,year
count,1499.000000,1499.000000,1499.000000,1499.000000,1499.000000,1499.000000,1499.000000,1499.000000,1499.000000
mean,76.016678,5.449229,6.108579,0.691825,66.661107,0.659324,0.320469,0.453302,2019.376918
std,43.887742,1.126387,2.499286,0.212825,7.673574,0.216246,0.172597,0.321830,2.858482
min,1.000000,1.721000,0.000000,0.000000,39.000000,0.000000,0.000000,0.000000,2015.000000
25%,38.000000,4.594450,4.377775,0.564165,62.000000,0.536015,0.196170,0.158835,2017.000000
50%,76.000000,5.472000,6.304790,0.738160,68.000000,0.690750,0.296280,0.345540,2019.000000
75%,114.000000,6.282250,8.049175,0.861890,72.000000,0.832070,0.429885,0.782680,2022.000000
max,158.000000,7.842100,10.000000,1.000000,85.000000,1.000000,1.000000,1.000000,2024.000000


In [74]:
"""" creaciíon de variables en meses para poder coincidir con el dataset de clima"""

months = pd.DataFrame({'month': range(1,13)})

df_happiness= (
    df_happiness
    .merge(months, how='cross')
)

In [ ]:
df_happiness.sample(10)

In [ ]:
df_happiness.month.nunique()

Comprobación de creación correcta de meses 

In [ ]:
df_happiness.duplicated(subset=["year"]).sum()

In [ ]:
df_happiness[df_happiness["country"] == "Turkey"]

In [ ]:
df_happiness.info()

# Carga datos de clima: en este caso debemos cargar y unificar en un sólo fichero. 

In [5]:
df_stations = pd.read_fwf('../DATOS/RAW/ghcnd-stations.txt',
    colspecs=[(0,11),(12,20),(21,30),(31,37)],
    names=["station_id","lat","lon","country"]
)

df_stations.info()

df_stations.sample(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 129657 entries, 0 to 129656
Data columns (total 4 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   station_id  129657 non-null  object 
 1   lat         129657 non-null  float64
 2   lon         129657 non-null  float64
 3   country     129657 non-null  float64
dtypes: float64(3), object(1)
memory usage: 4.0+ MB


,station_id,lat,lon,country
50765,SWE00137642,56.0400,15.8100,3.0
30827,CA008100467,47.8000,-64.8333,5.0
73086,US1MANT0006,41.2612,-70.0999,9.8
35024,GME00121642,50.7053,10.8589,518.0
36177,IDM00096035,3.5580,98.6720,34.7


Advertido de que con la información la columna "Country" no puedo extraer el pais, observo que en "station_id" se referecia en pais con las dos primeras letras. Se crea la columna "country_code". 

In [8]:
df_stations["country_code"] = df_stations["station_id"].str[:2]

df_stations.sample(5)

,station_id,lat,lon,country,country_code
74402,US1MIKN0094,42.8274,-85.5349,250.9,US
71941,US1KYHC0001,36.5391,-88.8319,133.2,US
14683,ASN00081010,-36.4500,144.8000,-999.9,AS
44499,MXN00020230,17.2667,-96.9000,1738.9,MX
8274,ASN00041017,-26.7422,150.6033,313.0,AS


In [9]:
df_stations["country_code"].unique()

array(['AC', 'AE', 'AF', 'AG', 'AJ', 'AL', 'AM', 'AO', 'AQ', 'AR', 'AS',
       'AU', 'AY', 'BA', 'BB', 'BC', 'BD', 'BE', 'BF', 'BG', 'BH', 'BK',
       'BL', 'BM', 'BN', 'BO', 'BP', 'BR', 'BU', 'BX', 'BY', 'CA', 'CB',
       'CD', 'CE', 'CF', 'CG', 'CH', 'CI', 'CJ', 'CK', 'CM', 'CO', 'CQ',
       'CS', 'CT', 'CU', 'CV', 'CW', 'CY', 'DA', 'DO', 'DR', 'EC', 'EG',
       'EI', 'EK', 'EN', 'ER', 'ES', 'ET', 'EU', 'EZ', 'FG', 'FI', 'FJ',
       'FK', 'FM', 'FP', 'FR', 'FS', 'GA', 'GB', 'GG', 'GH', 'GI', 'GL',
       'GM', 'GP', 'GQ', 'GR', 'GT', 'GV', 'GY', 'HO', 'HR', 'HU', 'IC',
       'ID', 'IN', 'IO', 'IR', 'IS', 'IT', 'IV', 'IZ', 'JA', 'JM', 'JN',
       'JO', 'JQ', 'JU', 'KE', 'KG', 'KN', 'KR', 'KS', 'KT', 'KU', 'KZ',
       'LA', 'LE', 'LG', 'LH', 'LI', 'LO', 'LQ', 'LT', 'LU', 'LY', 'MA',
       'MB', 'MC', 'MD', 'MF', 'MG', 'MI', 'MJ', 'MK', 'ML', 'MO', 'MP',
       'MQ', 'MR', 'MT', 'MU', 'MV', 'MX', 'MY', 'MZ', 'NC', 'NE', 'NF',
       'NG', 'NH', 'NI', 'NL', 'NN', 'NO', 'NP', 'N

Cargo datos del clima:

In [15]:
files = glob('../DATOS/RAW/clima_10años/*.csv')

Creación de dataset de clima con sus estaciones.

In [16]:

df_clima = pd.concat(
    [pd.read_csv(f, usecols=["ID","DATE","ELEMENT","DATA_VALUE"]) for f in files],
    ignore_index=True
)

df_clima.rename(columns={"ID":"station_id", "DATA_VALUE":"value"}, inplace=True)

In [ ]:
df_clima.sample(5)

In [17]:
df_climate = df_clima[df_clima["ELEMENT"].isin(["TMAX","TMIN","PRCP"])]

Normalización de la fecha a datatime

In [18]:
df_clima["DATE"] = pd.to_datetime(df_clima["DATE"].astype(str), format="%Y%m%d")

df_clima["year"] = df_clima["DATE"].dt.year
df_clima["month"] = df_clima["DATE"].dt.month



In [19]:
df_clima.loc[df_clima["ELEMENT"].isin(["TMAX","TMIN"]), "value"] /= 10
df_clima.loc[df_clima["ELEMENT"]=="PRCP", "value"] /= 10

/var/folders/3r/zlzms3_92977hqkp2r_c_vhw0000gn/T/ipykernel_44255/873827467.py:1: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '[ 16.8  14.8  17.4 ... -20.6 -14.4 -22.2]' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df_clima.loc[df_clima["ELEMENT"].isin(["TMAX","TMIN"]), "value"] /= 10


In [20]:
df_clima = df_clima.merge( df_stations,
    on="station_id",
    how="left"
)

In [ ]:
df_clima.info()

In [ ]:
df_clima.sample(5)

In [21]:
""""Filtrar variables climaticas"""

elements_keep = [
    "TAVG",
    "TMAX",
    "TMIN",
    "PRCP",
    "SNOW",
    "SNWD",
    "AWND"
]

df_clima = df_clima[df_clima["ELEMENT"].isin(elements_keep)]

In [22]:
df_clima.sample(10)

,station_id,DATE,ELEMENT,value,year,month,lat,lon,country,country_code
40489454,US1NYCR0005,2021-02-04,SNWD,432.0,2021,2,42.4516,-76.0437,313.9,US
83754739,USC00276368,2023-03-29,SNOW,0.0,2023,3,43.2142,-71.2469,161.5,US
24472455,US1KYMN0022,2020-08-31,PRCP,48.0,2020,8,37.5807,-84.3510,303.3,US
304103866,USC00036768,2017-09-18,PRCP,0.0,2017,9,33.9153,-92.8267,51.8,US
20940016,NOE00156763,2020-07-27,TMAX,23.5,2020,7,68.6481,15.2828,14.0,NO
300572126,USC00278614,2017-08-14,PRCP,0.0,2017,8,43.8644,-71.2664,164.9,US
12569864,US1NCHW0016,2020-05-04,PRCP,0.0,2020,5,35.5261,-82.9073,779.7,US
264708551,USC00299820,2015-12-01,TMIN,-13.3,2015,12,35.9478,-106.7469,2505.5,US
87658078,USW00053138,2023-05-06,TMAX,11.1,2023,5,39.0117,-114.2089,2016.9,US
45914793,ASN00013017,2021-03-29,TMIN,18.9,2021,3,-25.0341,128.3010,598.0,AS


In [23]:
df_clima["country_code"] = df_clima["station_id"].str[:2]

In [24]:
df_clima = df_clima[
    [
        "station_id",
        "DATE",
        "ELEMENT",
        "value",
        "year",
        "month",
        "country_code"
    ]
]
df_clima.info()

<class 'pandas.core.frame.DataFrame'>
Index: 259723971 entries, 0 to 314689503
Data columns (total 7 columns):
 #   Column        Dtype         
---  ------        -----         
 0   station_id    object        
 1   DATE          datetime64[ns]
 2   ELEMENT       object        
 3   value         float64       
 4   year          int32         
 5   month         int32         
 6   country_code  object        
dtypes: datetime64[ns](1), float64(1), int32(2), object(3)
memory usage: 13.5+ GB


In [25]:
df_clima.duplicated(subset=["station_id","year","month"]).sum()

np.int64(255834845)

In [ ]:
df_clima.sample(5)

# Creación de columnas con las variables filtradas y pasadas a un nuevo dataset mensual: 

In [26]:
df_clima_monthly = (
    df_clima
    .groupby(["country_code","year","month","ELEMENT"])["value"]
    .mean()
    .unstack()
    .reset_index()
)

In [27]:
df_clima_monthly = df_clima_monthly.rename(columns={
    "TMAX":"tmax_c",
    "TMIN":"tmin_c",
    "PRCP":"precip_mm"
})

In [28]:
""""Crear temperatura media"""

df_clima_monthly["temp_mean"] = (
    df_clima_monthly["tmax_c"] + df_clima_monthly["tmin_c"]
) / 2

creación de diccionario de paises: 

In [ ]:
df_clima_monthly.country_code.unique()

In [34]:


country_dict = {
"AC": "Antigua and Barbuda",
"AE": "United Arab Emirates",
"AF": "Afghanistan",
"AG": "Algeria",
"AJ": "Azerbaijan",
"AL": "Albania",
"AM": "Armenia",
"AO": "Angola",
"AQ": "American Samoa [United States]",
"AR": "Argentina",
"AS": "Australia",
"AU": "Austria",
"AY": "Antarctica",
"BA": "Bahrain",
"BB": "Barbados",
"BC": "Botswana",
"BD": "Bermuda [United Kingdom]",
"BE": "Belgium",
"BF": "Bahamas, The",
"BG": "Bangladesh",
"BH": "Belize",
"BK": "Bosnia and Herzegovina",
"BL": "Bolivia",
"BM": "Burma",
"BN": "Benin",
"BO": "Belarus",
"BP": "Solomon Islands",
"BR": "Brazil",
"BU": "Bulgaria",
"BX": "Brunei",
"BY": "Burundi",
"CA": "Canada",
"CB": "Cambodia",
"CD": "Chad",
"CE": "Sri Lanka",
"CF": "Congo (Brazzaville)",
"CG": "Congo (Kinshasa)",
"CH": "China",
"CI": "Chile",
"CJ": "Cayman Islands [United Kingdom]",
"CK": "Cocos (Keeling) Islands [Australia]",
"CM": "Cameroon",
"CO": "Colombia",
"CQ": "Northern Mariana Islands [United States]",
"CS": "Costa Rica",
"CT": "Central African Republic",
"CU": "Cuba",
"CV": "Cape Verde",
"CW": "Cook Islands [New Zealand]",
"CY": "Cyprus",
"DA": "Denmark",
"DO": "Dominica",
"DR": "Dominican Republic",
"EC": "Ecuador",
"EG": "Egypt",
"EI": "Ireland",
"EK": "Equatorial Guinea",
"EN": "Estonia",
"ER": "Eritrea",
"ES": "El Salvador",
"ET": "Ethiopia",
"EU": "Europa Island [France]",
"EZ": "Czech Republic",
"FG": "French Guiana [France]",
"FI": "Finland",
"FJ": "Fiji",
"FK": "Falkland Islands (Islas Malvinas) [United Kingdom]",
"FM": "Federated States of Micronesia",
"FP": "French Polynesia",
"FR": "France",
"FS": "French Southern and Antarctic Lands [France]",
"GA": "Gambia, The",
"GB": "Gabon",
"GG": "Georgia",
"GH": "Ghana",
"GI": "Gibraltar [United Kingdom]",
"GL": "Greenland [Denmark]",
"GM": "Germany",
"GP": "Guadeloupe [France]",
"GQ": "Guam [United States]",
"GR": "Greece",
"GT": "Guatemala",
"GV": "Guinea",
"GY": "Guyana",
"HO": "Honduras",
"HR": "Croatia",
"HU": "Hungary",
"IC": "Iceland",
"ID": "Indonesia",
"IN": "India",
"IO": "British Indian Ocean Territory [United Kingdom]",
"IR": "Iran",
"IS": "Israel",
"IT": "Italy",
"IV": "Cote D'Ivoire",
"IZ": "Iraq",
"JA": "Japan",
"JM": "Jamaica",
"JN": "Jan Mayen [Norway]",
"JO": "Jordan",
"JQ": "Johnston Atoll [United States]",
"JU": "Juan De Nova Island [France]",
"KE": "Kenya",
"KG": "Kyrgyzstan",
"KN": "Korea, North",
"KR": "Kiribati",
"KS": "Korea, South",
"KT": "Christmas Island [Australia]",
"KU": "Kuwait",
"KZ": "Kazakhstan",
"LA": "Laos",
"LE": "Lebanon",
"LG": "Latvia",
"LH": "Lithuania",
"LI": "Liberia",
"LO": "Slovakia",
"LQ": "Palmyra Atoll [United States]",
"LT": "Lesotho",
"LU": "Luxembourg",
"LY": "Libya",
"MA": "Madagascar",
"MB": "Martinique [France]",
"MC": "Macau S.A.R",
"MD": "Moldova",
"MF": "Mayotte [France]",
"MG": "Mongolia",
"MI": "Malawi",
"MJ": "Montenegro",
"MK": "Macedonia",
"ML": "Mali",
"MO": "Morocco",
"MP": "Mauritius",
"MQ": "Midway Islands [United States]",
"MR": "Mauritania",
"MT": "Malta",
"MU": "Oman",
"MV": "Maldives",
"MX": "Mexico",
"MY": "Malaysia",
"MZ": "Mozambique",
"NC": "New Caledonia [France]",
"NE": "Niue [New Zealand]",
"NF": "Norfolk Island [Australia]",
"NG": "Niger",
"NH": "Vanuatu",
"NI": "Nigeria",
"NL": "Netherlands",
"NN": "Sint Maarten",
"NO": "Norway",
"NP": "Nepal",
"NS": "Suriname",
"NU": "Nicaragua",
"NZ": "New Zealand",
"PA": "Paraguay",
"PC": "Pitcairn Islands [United Kingdom]",
"PE": "Peru",
"PK": "Pakistan",
"PL": "Poland",
"PM": "Panama",
"PO": "Portugal",
"PP": "Papua New Guinea",
"PS": "Palau",
"PU": "Guinea-Bissau",
"QA": "Qatar",
"RE": "Reunion [France]",
"RI": "Serbia",
"RM": "Marshall Islands",
"RO": "Romania",
"RP": "Philippines",
"RQ": "Puerto Rico [United States]",
"RS": "Russia",
"RW": "Rwanda",
"SA": "Saudi Arabia",
"SB": "Saint Pierre and Miquelon [France]",
"SE": "Seychelles",
"SF": "South Africa",
"SG": "Senegal",
"SH": "Saint Helena [United Kingdom]",
"SI": "Slovenia",
"SL": "Sierra Leone",
"SN": "Singapore",
"SP": "Spain",
"ST": "Saint Lucia",
"SU": "Sudan",
"SV": "Svalbard [Norway]",
"SW": "Sweden",
"SX": "South Georgia and the South Sandwich Islands [United Kingdom]",
"SY": "Syria",
"SZ": "Switzerland",
"TD": "Trinidad and Tobago",
"TE": "Tromelin Island [France]",
"TH": "Thailand",
"TI": "Tajikistan",
"TL": "Tokelau [New Zealand]",
"TN": "Tonga",
"TO": "Togo",
"TS": "Tunisia",
"TU": "Turkey",
"TV": "Tuvalu",
"TX": "Turkmenistan",
"TZ": "Tanzania",
"UC": "Curacao",
"UG": "Uganda",
"UK": "United Kingdom",
"UP": "Ukraine",
"US": "United States",
"UV": "Burkina Faso",
"UY": "Uruguay",
"UZ": "Uzbekistan",
"VE": "Venezuela",
"VM": "Vietnam",
"VQ": "Virgin Islands [United States]",
"WA": "Namibia",
"WF": "Wallis and Futuna [France]",
"WI": "Western Sahara",
"WQ": "Wake Island [United States]",
"WZ": "Swaziland",
"ZA": "Zambia",
"ZI": "Zimbabwe"
}


df_clima_monthly["country"] = df_clima_monthly["country_code"].map(country_dict)
        

In [41]:
df_clima_monthly.sample(10)

ELEMENT,country_code,year,month,AWND,precip_mm,SNOW,SNWD,TAVG,tmax_c,tmin_c,temp_mean,country
2931,CA,2023,9,NaN,1.713416,0.013834,2.432566,132.357840,19.291505,7.854217,13.572861,Canada
10986,LY,2017,10,NaN,5.516667,NaN,NaN,218.340541,26.127397,16.777344,21.452371,Libya
6906,GB,2024,12,NaN,5.572941,NaN,NaN,268.955556,30.840741,23.536000,27.188370,Gabon
2265,BN,2019,9,NaN,8.880882,NaN,NaN,259.411111,30.548521,22.520408,26.534464,Benin
10622,LH,2022,7,NaN,5.701639,NaN,NaN,178.681818,22.682353,13.130864,17.906609,Lithuania
4469,CU,2023,2,NaN,3.950000,NaN,NaN,253.642857,30.907143,20.164286,25.535714,Cuba
5412,EK,2024,6,NaN,4.831429,NaN,NaN,268.533333,30.700000,24.550000,27.625000,Equatorial Guinea
11866,MK,2022,3,NaN,0.367391,NaN,10.000000,46.393617,10.814634,-0.542500,5.136067,Macedonia
1793,BG,2023,4,NaN,2.356856,NaN,NaN,289.876667,35.045455,23.421610,29.233532,Bangladesh
3945,CM,2022,5,NaN,7.838000,NaN,NaN,287.522124,33.439130,20.650000,27.044565,Cameroon


In [42]:
df_clima_monthly.info()



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19838 entries, 0 to 19837
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   country_code  19838 non-null  object 
 1   year          19838 non-null  int32  
 2   month         19838 non-null  int32  
 3   AWND          866 non-null    float64
 4   precip_mm     17405 non-null  float64
 5   SNOW          925 non-null    float64
 6   SNWD          5053 non-null   float64
 7   TAVG          19294 non-null  float64
 8   tmax_c        17871 non-null  float64
 9   tmin_c        18567 non-null  float64
 10  temp_mean     17753 non-null  float64
 11  country       19838 non-null  object 
dtypes: float64(8), int32(2), object(2)
memory usage: 1.7+ MB


In [43]:
df_clima_monthly.isnull().sum()

ELEMENT
country_code        0
year                0
month               0
AWND            18972
precip_mm        2433
SNOW            18913
SNWD            14785
TAVG              544
tmax_c           1967
tmin_c           1271
temp_mean        2085
country             0
dtype: int64

# Tratamiento de nulos:

In [44]:
""""Se elimina AWND debido a la elevada proporción de valores faltantes (>95%), lo que compromete la fiabilidad del análisis."""

df_clima_monthly = df_clima_monthly.drop(columns=["AWND"])

In [45]:
""""Los valores faltantes en variables de nieve se interpretan como ausencia de nieve y se sustituyen por 0."""

df_clima_monthly["SNOW"] = df_clima_monthly["SNOW"].fillna(0)
df_clima_monthly["SNWD"] = df_clima_monthly["SNWD"].fillna(0)


In [47]:
""""Imputación de precipitación: se utiliza la mediana por país para preservar la distribución y reducir el impacto de valores atípicos."""

df_clima_monthly["precip_mm"] = df_clima_monthly.groupby("country_code")["precip_mm"].transform(
    lambda x: x.fillna(x.median()) if x.notna().any() else x
)

df_clima_monthly["precip_mm"] = df_clima_monthly["precip_mm"].fillna(
    df_clima_monthly["precip_mm"].median()
)

In [48]:
df_clima_monthly.groupby("country_code")["precip_mm"].apply(lambda x: x.isna().all())

country_code
AE    False
AF    False
AG    False
AJ    False
AL    False
      ...  
WI    False
WQ    False
WZ    False
ZA    False
ZI    False
Name: precip_mm, Length: 201, dtype: bool

La variable precip_mm presenta valores faltantes. Se imputan utilizando la mediana de precipitación por país, lo que permite preservar las características climáticas propias de cada región y reducir la influencia de valores extremos. En los casos en que un país no presenta registros válidos, se utiliza la mediana global del dataset.

In [49]:
df_clima_monthly["tmax_c"] = df_clima_monthly.groupby(
    ["country_code", "month"]
)["tmax_c"].transform(
    lambda x: x.fillna(x.median()) if x.notna().any() else x
)

df_clima_monthly["tmin_c"] = df_clima_monthly.groupby(
    ["country_code", "month"]
)["tmin_c"].transform(
    lambda x: x.fillna(x.median()) if x.notna().any() else x
)

In [50]:
df_clima_monthly["tmax_c"] = df_clima_monthly["tmax_c"].fillna(df_clima_monthly["tmax_c"].median())
df_clima_monthly["tmin_c"] = df_clima_monthly["tmin_c"].fillna(df_clima_monthly["tmin_c"].median())

In [51]:
df_clima_monthly["temp_mean"] = (df_clima_monthly["tmax_c"] + df_clima_monthly["tmin_c"]) / 2

In [52]:
df_clima_monthly.isnull().sum()

ELEMENT
country_code      0
year              0
month             0
precip_mm         0
SNOW              0
SNWD              0
TAVG            544
tmax_c            0
tmin_c            0
temp_mean         0
country           0
dtype: int64

In [53]:
df_clima_monthly["TAVG"] = df_clima_monthly["TAVG"].fillna(df_clima_monthly["temp_mean"])

In [54]:
df_clima_monthly = df_clima_monthly.drop(columns="TAVG")

Creación de nuevas columnas:

In [55]:
"""" creación de rango de temperatura """

df_clima_monthly["temp_range"] = df_clima_monthly["tmax_c"] - df_clima_monthly["tmin_c"]

In [56]:
"""" creacicón de estacionalidad   """

def season(m):
    if m in [12,1,2]:
        return "winter"
    elif m in [3,4,5]:
        return "spring"
    elif m in [6,7,8]:
        return "summer"
    else:
        return "autumn"

df_clima_monthly["season"] = df_clima_monthly["month"].apply(season)

In [58]:
df_clima_monthly["temp_anomaly"] = df_clima_monthly["temp_mean"] - df_clima_monthly.groupby(
    ["country_code","month"]
)["temp_mean"].transform("mean")

In [59]:
df_clima_monthly["precip_anomaly"] = df_clima_monthly["precip_mm"] - df_clima_monthly.groupby(
    ["country_code","month"]
)["precip_mm"].transform("mean")

In [60]:
df_clima_monthly.sample(10)

ELEMENT,country_code,year,month,precip_mm,SNOW,SNWD,tmax_c,tmin_c,temp_mean,country,temp_range,season,temp_anomaly,precip_anomaly
16212,SG,2016,3,5.954007,0.0,0.0,37.378082,20.464055,28.921069,Senegal,16.914027,spring,-0.430386,3.395404
9843,KS,2016,4,8.777778,0.0,0.0,17.966667,7.797436,12.882051,"Korea, South",10.169231,spring,-1.534187,-0.374474
4634,CW,2019,9,0.557895,0.0,0.0,26.940000,19.425000,23.182500,Cook Islands [New Zealand],7.515000,autumn,0.000282,-2.140139
8269,ID,2015,9,15.530769,0.0,0.0,31.961466,22.743780,27.352623,Indonesia,9.217686,autumn,-0.250148,-0.411054
1755,BG,2020,2,0.216364,0.0,0.0,28.813333,16.026374,22.419853,Bangladesh,12.786960,winter,-0.496982,-1.391807
14453,PO,2015,7,0.823529,0.0,0.0,29.281287,18.669538,23.975413,Portugal,10.611748,summer,-0.077713,-0.092393
2528,BP,2024,6,4.267059,0.0,0.0,32.837143,23.978333,28.407738,Solomon Islands,8.858810,summer,0.472157,-0.897119
15427,RP,2020,11,12.702368,0.0,0.0,30.413701,23.097970,26.755835,Philippines,7.315732,autumn,-0.330926,1.687652
8107,HU,2020,10,3.690647,0.0,0.0,16.943713,7.881349,12.412531,Hungary,9.062363,autumn,-0.042190,1.741316
11561,MG,2022,6,5.230070,0.0,41.0,24.756472,9.840461,17.298466,Mongolia,14.916011,summer,0.541418,-0.521621


In [61]:
df_clima_monthly.columns

Index(['country_code', 'year', 'month', 'precip_mm', 'SNOW', 'SNWD', 'tmax_c',
       'tmin_c', 'temp_mean', 'country', 'temp_range', 'season',
       'temp_anomaly', 'precip_anomaly'],
      dtype='object', name='ELEMENT')

In [62]:
df_clima_monthly.duplicated(subset=["country_code","year","month"]).sum()

np.int64(0)

In [ ]:
df_clima_monthly.info()

In [ ]:
df_happiness.info()

In [63]:
df_clima_monthly = df_clima_monthly.drop(columns="country_code")

In [ ]:
df_clima_monthly.info()

In [ ]:
df_clima_monthly.columns

El dataset climático fue agregado en dos etapas: primero a nivel mensual por estación meteorológica y posteriormente a nivel país-mes-año mediante promedios, con el objetivo de evitar duplicados durante la integración con el dataset de felicidad.

In [64]:
df_clima_monthly.duplicated(subset=["country","year","month"]).sum()

np.int64(0)

In [ ]:
df_clima_monthly[df_clima_monthly.duplicated(subset=["country","year","month"], keep=False)].sort_values(["country","year","month"])

In [65]:
df_clima_monthly = df_clima_monthly.groupby(
["country","year","month", "season"],
as_index=False
).mean(numeric_only=True)

In [66]:
df_clima_monthly.duplicated(subset=["country","year","month"]).sum()

np.int64(0)

In [ ]:
df_clima_monthly.info()

In [ ]:
df_happiness.info()

In [75]:
set_h = set(zip(df_happiness.country, df_happiness.year, df_happiness.month))
set_c = set(zip(df_clima_monthly.country, df_clima_monthly.year, df_clima_monthly.month))

len(set_h - set_c)  # happiness sin clima
len(set_c - set_h)  # clima sin happiness

6487

In [76]:
set(df_happiness.country) - set(df_clima_monthly.country)

{'Argelia',
 'Bhutan',
 'Burundi',
 'Comoros',
 'Congo',
 'Czechia',
 'Djibouti',
 'Eswatini',
 'Gambia',
 'Guatemala',
 'Haiti',
 'Hong Kong',
 'Hong Kong S.A.R. of China',
 'Ivory Coast',
 'Kosovo',
 'Lesotho',
 'Myanmar',
 'North Cyprus',
 'North Macedonia',
 'Palestinian Territories',
 'Panama',
 'Puerto Rico',
 'Somalia',
 'Somaliland region',
 'South Korea',
 'South Sudan',
 'State of Palestine',
 'Taiwan',
 'Taiwan Province of China',
 'Trinidad & Tobago',
 'Turkiye',
 'Uganda',
 'Yemen'}

In [77]:
set(df_clima_monthly.country) - set(df_happiness.country)

{'American Samoa [United States]',
 'Antarctica',
 'Bahamas, The',
 'Bermuda [United Kingdom]',
 'Brunei',
 'Burma',
 'Cape Verde',
 'Cayman Islands [United Kingdom]',
 'Christmas Island [Australia]',
 'Cocos (Keeling) Islands [Australia]',
 'Cook Islands [New Zealand]',
 "Cote D'Ivoire",
 'Cuba',
 'Equatorial Guinea',
 'Eritrea',
 'Europa Island [France]',
 'Falkland Islands (Islas Malvinas) [United Kingdom]',
 'Federated States of Micronesia',
 'Fiji',
 'French Guiana [France]',
 'French Polynesia',
 'French Southern and Antarctic Lands [France]',
 'Gambia, The',
 'Gibraltar [United Kingdom]',
 'Greenland [Denmark]',
 'Guadeloupe [France]',
 'Guam [United States]',
 'Guinea-Bissau',
 'Guyana',
 'Jan Mayen [Norway]',
 'Juan De Nova Island [France]',
 'Kiribati',
 'Korea, North',
 'Korea, South',
 'Macau S.A.R',
 'Marshall Islands',
 'Martinique [France]',
 'Mayotte [France]',
 'Midway Islands [United States]',
 'New Caledonia [France]',
 'Niue [New Zealand]',
 'Norfolk Island [Austral

In [78]:
h_countries = set(df_happiness.country)
c_countries = set(df_clima_monthly.country)

h_countries - c_countries

{'Argelia',
 'Bhutan',
 'Burundi',
 'Comoros',
 'Congo',
 'Czechia',
 'Djibouti',
 'Eswatini',
 'Gambia',
 'Guatemala',
 'Haiti',
 'Hong Kong',
 'Hong Kong S.A.R. of China',
 'Ivory Coast',
 'Kosovo',
 'Lesotho',
 'Myanmar',
 'North Cyprus',
 'North Macedonia',
 'Palestinian Territories',
 'Panama',
 'Puerto Rico',
 'Somalia',
 'Somaliland region',
 'South Korea',
 'South Sudan',
 'State of Palestine',
 'Taiwan',
 'Taiwan Province of China',
 'Trinidad & Tobago',
 'Turkiye',
 'Uganda',
 'Yemen'}

In [ ]:
c_countries - h_countries

In [79]:
country_map = {
    "Congo (Brazzaville)": "Congo",
    "Congo (Kinshasa)": "Democratic Republic of the Congo",
    "Czechia": "Czech Republic",
    "Macedonia": "North Macedonia",
    "Turkiye": "Turkey",
    "Trinidad & Tobago": "Trinidad and Tobago",
    "State of Palestine": "Palestine",
    "Palestinian Territories": "Palestine"
}

df_happiness["country"] = df_happiness["country"].replace(country_map)

In [80]:
country_map_2 = {
    "Hong Kong S.A.R. of China": "Hong Kong",
    "Taiwan Province of China": "Taiwan",
    "Trinidad & Tobago": "Trinidad and Tobago"
}

df_happiness["country"] = df_happiness["country"].replace(country_map_2)

In [81]:
country_map_extra = {
    "Argelia": "Algeria",
    "Panama": "Panama",
    "Hong Kong": "Hong Kong SAR",
    "Taiwan": "Taiwan, Province of China",
    "Palestine": "West Bank and Gaza",
    "Gambia": "Gambia, The"
}

In [82]:
exclude_countries = [
    "Kosovo",
    "Somaliland region",
    "North Cyprus"
]

In [83]:
set(df_happiness.country) - set(df_clima_monthly.country)

{'Argelia',
 'Bhutan',
 'Burundi',
 'Comoros',
 'Congo',
 'Democratic Republic of the Congo',
 'Djibouti',
 'Eswatini',
 'Gambia',
 'Guatemala',
 'Haiti',
 'Hong Kong',
 'Ivory Coast',
 'Kosovo',
 'Lesotho',
 'Myanmar',
 'North Cyprus',
 'North Macedonia',
 'Palestine',
 'Panama',
 'Puerto Rico',
 'Somalia',
 'Somaliland region',
 'South Korea',
 'South Sudan',
 'Taiwan',
 'Uganda',
 'Yemen'}

In [84]:
df_clima_monthly.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19838 entries, 0 to 19837
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   country         19838 non-null  object 
 1   year            19838 non-null  int32  
 2   month           19838 non-null  int32  
 3   season          19838 non-null  object 
 4   precip_mm       19838 non-null  float64
 5   SNOW            19838 non-null  float64
 6   SNWD            19838 non-null  float64
 7   tmax_c          19838 non-null  float64
 8   tmin_c          19838 non-null  float64
 9   temp_mean       19838 non-null  float64
 10  temp_range      19838 non-null  float64
 11  temp_anomaly    19838 non-null  float64
 12  precip_anomaly  19838 non-null  float64
dtypes: float64(9), int32(2), object(2)
memory usage: 1.8+ MB


In [85]:
df_clima_monthly.sample(5)

ELEMENT,country,year,month,season,precip_mm,SNOW,SNWD,tmax_c,tmin_c,temp_mean,temp_range,temp_anomaly,precip_anomaly
19210,Vietnam,2015,5,spring,4.140433,0.0,0.0,35.226923,26.070144,30.648533,9.156779,1.833344,-1.200434
6695,Gabon,2024,7,summer,0.186667,0.0,0.0,27.483333,21.561224,24.522279,5.922109,-0.144904,-2.403082
19305,Vietnam,2024,9,autumn,16.124672,0.0,0.0,32.265482,25.124506,28.694994,7.140976,0.405786,3.737302
6361,French Guiana [France],2020,2,winter,2.262069,0.0,0.0,30.243750,23.946154,27.094952,6.297596,0.349664,-8.889616
1000,Azerbaijan,2015,11,autumn,29.333333,0.0,0.0,13.459567,7.129730,10.294648,6.329837,0.612593,17.669451


In [86]:
df_happiness.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17988 entries, 0 to 17987
Data columns (total 12 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   ranking                       17988 non-null  int64  
 1   country                       17988 non-null  object 
 2   regional_indicator            17988 non-null  object 
 3   happiness_score               17988 non-null  float64
 4   gdp_per_capita                17988 non-null  float64
 5   social_support                17988 non-null  float64
 6   healthy_life_expectancy       17988 non-null  int64  
 7   freedom_to_make_life_choices  17988 non-null  float64
 8   generosity                    17988 non-null  float64
 9   perceptions_of_corruption     17988 non-null  float64
 10  year                          17988 non-null  int64  
 11  month                         17988 non-null  int64  
dtypes: float64(6), int64(4), object(2)
memory usage: 1.6+ MB


In [87]:
df_happiness.country.value_counts()

country
Switzerland               120
Hungary                   120
North Macedonia           120
Albania                   120
Bosnia and Herzegovina    120
                         ... 
Suriname                   24
Eswatini                   24
Puerto Rico                12
Djibouti                   12
Oman                       12
Name: count, Length: 167, dtype: int64

In [88]:
df_clima_monthly.country.value_counts()

country
Kyrgyzstan                      103
Jamaica                         103
Maldives                        103
Mali                            103
Malta                           103
                               ... 
Juan De Nova Island [France]     23
Guadeloupe [France]              17
Niue [New Zealand]               16
Martinique [France]              13
Eritrea                           1
Name: count, Length: 201, dtype: int64

In [89]:
# Guardar los datasets limpios para su posterior análisis y modelado.

df_clima_monthly_clean = df_clima_monthly
df_happiness_clean = df_happiness

df_happiness_clean.to_csv("../DATOS/Output/df_happiness_clean.csv", index=False)

df_clima_monthly_clean.to_csv("../DATOS/Output/df_clima_monthly_clean.csv", index=False)

In [90]:
set_happy = set(df_happiness["country"].unique())
set_clima = set(df_clima_monthly["country"].unique())

missing = set_happy - set_clima
len(missing), list(missing)[:]

(28,
 ['Comoros',
  'North Cyprus',
  'Puerto Rico',
  'Guatemala',
  'North Macedonia',
  'Somalia',
  'South Sudan',
  'Uganda',
  'Bhutan',
  'Panama',
  'Argelia',
  'Burundi',
  'Haiti',
  'Congo',
  'Eswatini',
  'Djibouti',
  'South Korea',
  'Democratic Republic of the Congo',
  'Taiwan',
  'Gambia',
  'Myanmar',
  'Yemen',
  'Hong Kong',
  'Kosovo',
  'Palestine',
  'Lesotho',
  'Somaliland region',
  'Ivory Coast'])

In [91]:
df_final = pd.merge(
    df_happiness,
    df_clima_monthly,
    on=["country", "year", "month"],
    how="inner"
)

df_final.shape

(13189, 22)

In [92]:
df_final.columns

Index(['ranking', 'country', 'regional_indicator', 'happiness_score',
       'gdp_per_capita', 'social_support', 'healthy_life_expectancy',
       'freedom_to_make_life_choices', 'generosity',
       'perceptions_of_corruption', 'year', 'month', 'season', 'precip_mm',
       'SNOW', 'SNWD', 'tmax_c', 'tmin_c', 'temp_mean', 'temp_range',
       'temp_anomaly', 'precip_anomaly'],
      dtype='object')

In [93]:
df_final.isnull().sum()

ranking                         0
country                         0
regional_indicator              0
happiness_score                 0
gdp_per_capita                  0
social_support                  0
healthy_life_expectancy         0
freedom_to_make_life_choices    0
generosity                      0
perceptions_of_corruption       0
year                            0
month                           0
season                          0
precip_mm                       0
SNOW                            0
SNWD                            0
tmax_c                          0
tmin_c                          0
temp_mean                       0
temp_range                      0
temp_anomaly                    0
precip_anomaly                  0
dtype: int64

In [94]:
df_final.sample(5)

,ranking,country,regional_indicator,happiness_score,gdp_per_capita,social_support,healthy_life_expectancy,freedom_to_make_life_choices,generosity,perceptions_of_corruption,...,season,precip_mm,SNOW,SNWD,tmax_c,tmin_c,temp_mean,temp_range,temp_anomaly,precip_anomaly
3467,77,Croatia,Central and Eastern Europe,5.2930,6.53506,0.60102,68,0.38856,0.29592,0.09283,...,autumn,5.056429,0.000000,0.000000,12.431931,5.686829,9.059380,6.745101,-1.398475,0.463825
5927,18,United States,North America and ANZ,6.9396,8.94129,0.90774,68,0.77114,0.52323,0.71437,...,autumn,3.016761,0.297488,0.828331,24.586990,10.873829,17.730409,13.713161,-0.722518,0.368759
548,50,Italy,Western Europe,5.9480,7.40136,0.85419,73,0.39174,0.28676,0.94744,...,autumn,2.104071,0.000000,30.333333,24.492691,16.214727,20.353709,8.277964,-0.121759,-0.047882
5511,133,Ukraine,Commonwealth of Independent States,4.3322,4.87103,0.85639,71,0.28197,0.33024,0.02155,...,winter,2.746261,0.000000,180.128144,-0.516990,-6.687028,-3.602009,6.170038,-1.607778,0.134576
12223,95,Guinea,Sub-Saharan Africa,5.0233,3.88022,0.38454,62,0.60381,0.52419,0.18683,...,spring,8.460714,0.000000,0.000000,35.182524,23.325926,29.254225,11.856598,0.747139,1.576025


Integración de datasets
Para el análisis se integraron dos fuentes de datos:
World Happiness Dataset, que contiene indicadores de bienestar y felicidad por país.
Dataset climático, con variables meteorológicas agregadas por país, año y mes.
Dado que ambos datasets utilizan diferentes convenciones para nombrar algunos países, fue necesario realizar un proceso previo de armonización de nombres. Para ello se construyó un diccionario de equivalencias que permitió estandarizar los nombres y garantizar la correcta correspondencia entre registros.
Posteriormente se realizó la integración de los datasets mediante un merge utilizando las variables country, year y month como claves comunes.
El tipo de unión utilizado fue inner join, para preservar sólo las coincidencias, ya que durante el proceso de integración de datos se detectaron valores nulos en las variables climáticas.
El análisis mostró que estos nulos no eran aleatorios, sino que correspondían a países completos sin información climática disponible en la fuente utilizada (por ejemplo, Panamá, Uganda o Kosovo).
Dado que no existía información histórica suficiente para estos países, se descartó la imputación de valores para evitar la introducción de sesgos.
En consecuencia, se procedió a eliminar dichos registros, garantizando así la consistencia y fiabilidad del análisis posterior.
En aquellos casos en los que no existían datos climáticos correspondientes, los valores resultantes se mantuvieron como valores nulos (NaN).
Este enfoque permite preservar la mayor cantidad de observaciones posibles del dataset principal, evitando la pérdida de información relevante para el análisis.

### El dataset final combina:

# Variables de felicidad
ranking
happiness_score
gdp_per_capita
social_support
healthy_life_expectancy
freedom_to_make_life_choices
generosity
perceptions_of_corruption


# Variables temporales
country 
year 
month


# Variables climáticas
precip_mm 
SNOW 
SNWD 
tmax_c 
tmin_c 
temp_mean 
temp_range 
snow_flag 
rain_flag 
temp_anomaly 
precip_anomaly

In [ ]:
df_final[["country","year","month"]].duplicated().sum()

In [ ]:
df_final.isna().sum()

In [95]:
df_final = df_final.dropna(subset=['regional_indicator'])

In [96]:
df_final.to_csv("../DATOS/Output/df_final.csv", index=False, float_format='%.4f')

In [ ]:
df_final.columns

In [ ]:
df_final.shape
df_final.info()
df_final.describe()
df_final.isnull().sum()

# Análisis de varibles únicas

In [ ]:
# Felicidad 

sns.histplot(df_final['happiness_score'], kde=True)

In [ ]:
# Temperatura 

sns.histplot(df_final['temp_mean'], kde=True)

In [ ]:
# Precipitación

sns.histplot(df_final['precip_mm'], kde=True)

# Análisis por país / región

In [ ]:
# Top países más felices

df_final.groupby('country')['happiness_score'].mean().sort_values(ascending=False).head(10)

In [ ]:
# Región: 

df_final.groupby('regional_indicator')['happiness_score'].mean().sort_values(ascending=False).head(10)

In [ ]:
# Análisis temporal 

df_final.groupby('year')['happiness_score'].mean().plot()

In [ ]:
df_final.groupby('month')['temp_mean'].mean().plot()

In [ ]:
# Temperatura vs. Felicidad

sns.scatterplot(x='temp_mean', y='happiness_score', data=df_final)

In [ ]:
corr = df_final.corr(numeric_only=True)
sns.heatmap(corr, annot=True, cmap='coolwarm')

El clima tiene una influencia limitada sobre la felicidad en comparación con factores estructurales como el nivel económico o el bienestar social.

Las variables de temperatura son redundantes (miden lo mismo en esencia)

Las desviaciones climáticas puntuales no parecen impactar significativamente en el bienestar percibido.

La percepción de libertad individual también juega un papel relevante en la felicidad.

El análisis de correlación revela que la felicidad presenta una fuerte relación con variables socioeconómicas como el PIB per cápita, el apoyo social y la esperanza de vida saludable.
En contraste, las variables climáticas, como la temperatura media o la precipitación, muestran correlaciones débiles, lo que sugiere que su impacto sobre la felicidad es limitado.
Asimismo, las anomalías climáticas no parecen influir de manera significativa en el bienestar percibido.
Estos resultados indican que la felicidad es un fenómeno multifactorial, donde los factores estructurales tienen un peso considerablemente mayor que las condiciones climáticas.

In [ ]:
plt.figure(figsize=(8,5))
sns.scatterplot(
    x='temp_mean',
    y='happiness_score',
    data=df_final,
    alpha=0.6
)

sns.regplot(
    x='temp_mean',
    y='happiness_score',
    data=df_final,
    scatter=False,
    color='red'
)

plt.title('Average Temperature vs Happiness Score')
plt.xlabel('Temperature (°C)')
plt.ylabel('Happiness Score')

plt.show()

In [ ]:
sns.scatterplot(
    x='gdp_per_capita',
    y='happiness_score',
    hue='regional_indicator',
    data=df_final,
    alpha=0.6
)

In [ ]:
# comparación de niveles de felicidad vs clima en Latinoamérica

df_final[df_final['regional_indicator'] == 'Latin America and Caribbean'].groupby('temp_mean')['happiness_score'].mean().plot()

In [ ]:
df_final[df_final['regional_indicator'] == 'Central and Eastern Europe'].groupby('temp_mean')['happiness_score'].mean().plot()

En Paises de bajos ingresos si que se nota una diferencia y mayor impacto de la situación climatica en los niveles de felicidad

Ampliación de Dataset : generación de escenarios simulados para ampliar el dataset final.  Se descarta esta opción luego de obtener la confirmación de que puede realizarse el trabajo con un menor número de filas que las indicadas en los requisitos mínimos. 

In [ ]:
scenarios = {
    'normal': 0,
    'heat': 2,
    'cold': -2
}

df_expanded = pd.concat([
    df_final.assign(
        scenario=key,
        temp_mean=df_final['temp_mean'] + val
    )
    for key, val in scenarios.items()
])

In [ ]:
df_expanded.shape

In [ ]:
sns.boxplot(x='scenario', y='happiness_score', data=df_expanded)